In [ ]:
%config IPCompleter.use_jedi = False
%pdb off
%load_ext autoreload
%autoreload 3

%matplotlib qt
import matplotlib.pyplot as plt

from pathlib import Path
import spikeinterface.extractors as se
import spikeinterface.full as si
from neuropy.core.session.init_from_raw_data import windows_to_wsl_path_if_needed, find_first_file_rglob


In [ ]:
import spikeinterface.full as si


## Bapun Format:
# basedir = Path('/home/halechr/FastData/Bapun/RatS/Day1Openfield') # Linux
basedir = windows_to_wsl_path_if_needed(Path('W:/Data/Bapun/RatS/Day1Openfield')) # Windows
# basedir = Path("/mnt/w/Data/Bapun/RatS/Day1Openfield") # Apogee WSL
# basedir = Path('/Volumes/iNeo/Data/Bapun/Day5TwoNovel') # MacOS

# n_channels: int = 200
n_channels: int = 195
dat_file_sampling_rate: int = 30000
eeg_sampling_rate: int = 1250
basename: str = 'RatS-Day1Openfield'
excluded_data_datetimes = ['2020-11-25_10-24-24', '2020-11-25_15-06-02']

xml_path: Path = find_first_file_rglob(basedir, '*.xml', recursive=False)
xml_path
print(f'xml_path: {xml_path}')


In [ ]:
concatenated_dat_file: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.dat").resolve())
print(f"concatenated_dat_file: {concatenated_dat_file.as_posix()}")
assert concatenated_dat_file.exists(), f"concatenated_dat_file: {concatenated_dat_file} does not exist."


In [ ]:
spiking_circus_dir: Path = windows_to_wsl_path_if_needed(basedir.joinpath('spyk-circ', basename).resolve())
assert spiking_circus_dir.exists(), f"spiking_circus_dir: {spiking_circus_dir} does not exist!"
# 1. Map the raw binary file (doesn't load into RAM yet)
# se.read_phy
spiking_circus_loaded = se.read_spykingcircus(spiking_circus_dir)

spiking_circus_loaded

In [ ]:
n_channels

In [ ]:
recording = se.read_binary(
    file_paths=concatenated_dat_file.as_posix(),
    sampling_frequency=dat_file_sampling_rate,
    num_channels=n_channels,
    dtype="int16",
)
recording

In [ ]:
from probeinterface.io import read_prb

prb_path: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}.prb").resolve())
probe = read_prb(prb_path).probes[0]
recording = recording.set_probe(probe, in_place=False)
recording_filtered = si.bandpass_filter(recording)
recording_filtered


In [ ]:
sorting = spiking_circus_loaded

job_kwargs = dict(n_jobs=-1, progress_bar=True, chunk_duration="1s")

sorting_analyzer_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("spyk-circ", f"{basename}-sorting_analyzer").resolve())
sorting_analyzer = si.create_sorting_analyzer(sorting, recording_filtered, format="binary_folder", folder=sorting_analyzer_folder.as_posix())
sorting_analyzer.compute("random_spikes", method="uniform", max_spikes_per_unit=500)
sorting_analyzer.compute("waveforms", **job_kwargs)
sorting_analyzer.compute("templates", **job_kwargs)
sorting_analyzer.compute("noise_levels")
sorting_analyzer.compute("unit_locations", method="monopolar_triangulation")
sorting_analyzer.compute("isi_histograms")
sorting_analyzer.compute("correlograms", window_ms=100, bin_ms=5.)
sorting_analyzer.compute("principal_components", n_components=3, mode='by_channel_global', whiten=True, **job_kwargs)
sorting_analyzer.compute("quality_metrics", metric_names=["snr", "firing_rate"])
sorting_analyzer.compute("template_similarity")
sorting_analyzer.compute("spike_amplitudes", **job_kwargs)

In [ ]:
from spikeinterface.sorters import installed_sorters, run_sorter

installed_sorters() # ['kilosort4', 'lupin', 'simple', 'spykingcircus2', 'tridesclous2']

sorting_outputs_folder: Path = windows_to_wsl_path_if_needed(basedir.joinpath("SORTING").resolve()) # , f"{basename}-sorting_analyzer"
sorting_outputs_folder.mkdir(exist_ok=True)
print(f'sorting_outputs_folder: {sorting_outputs_folder}')

# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder=sorting_outputs_folder.joinpath("folder_SC2"), remove_existing_folder=True, delete_output_folder=True)


In [ ]:

# run Tridesclous
sorting_TDC = run_sorter(sorter_name="tridesclous", recording=recording, folder="/folder_TDC")
sorting_TDC.save(folder='/path/to/tridescloud_sorting_output')

# run Kilosort4
sorting_KS2_5 = run_sorter(sorter_name="kilosort4", recording=recording, folder="/folder_KS4")



In [ ]:
sorting = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2tmp/folder")

In [ ]:
# run SpykingCircus2
sorting_SC2 = run_sorter(sorter_name="spykingcircus2", recording=recording, folder="/folder_SC2")